# Patrón Estructural: Adapter (Adaptador)

Ahora vamos a plantear un problema con respecto a como un restaurante gestiona sus reservas.

Se utiliza un sistema interno que espera trabajar con un método para reservar una mesa. Sin embargo, el establecimiento también quiere utilizar diferentes plataformas externas de reservas. El problema es que cada plataforma tiene una forma diferente de realizar una reserva.

## Sin patrón Adapter ❌

Si no se aplica el patrón Adapter, la incompatibilidad entre interfaces queda explícita puesto que cada sistema (propio y externos) tiene su propia implementacion para manejar las reservas.

In [10]:
from abc import ABC, abstractmethod

class ReservationSystem(ABC):
    @abstractmethod
    def reserve_table(self, customer: str, date: str, people: int) -> str:
        ...


class OpenTableAPI:
    def book_table(self, customer_name: str, reservation_date: str, guests: int ) -> str:
        return ( f"Reserva realizada en OpenTable para {customer_name}, {guests} personas el {reservation_date}")

class GoogleReservationsAPI:
    def make_reservation(self, name: str, date: str, number_of_people: int) -> str:
        return ( f"Reserva realizada en Google para {name}, {number_of_people} personas el {date}")


cliente = "Carlos"
fecha = "2026-09-15"
personas = 4

sistema = OpenTableAPI()
reserva = sistema.reserve_table(cliente, fecha, personas)

sistema = GoogleReservationsAPI()
reserva = sistema.reserve_table(cliente, fecha, personas)

print(reserva)

# OpenTableAPI y GoogleReservationsAPI no poseen el metodo reserve_table
# propio del sistema.


AttributeError: 'OpenTableAPI' object has no attribute 'reserve_table'

### Con patrón Adapter ✅

Aplicando el patrón, se crea una interfaz común que representa la forma en que el restaurante desea realizar una reserva.

Los sistemas externos no necesitan ser modificados. En su lugar, se crean adaptadores que traducen la interfaz del restaurante a la interfaz específica de cada plataforma.

In [9]:
class OpenTableAdapter(ReservationSystem):
    def __init__(self, api: OpenTableAPI) -> None:
        self.api = api

    def reserve_table(self, customer: str, date: str, people: int) -> str:
        return self.api.book_table(customer, date, guests=people)

class GoogleReservationsAdapter(ReservationSystem):
    def __init__(self, api: GoogleReservationsAPI) -> None:
        self.api = api

    def reserve_table(self, customer: str, date: str, people: int) -> str:
        return self.api.make_reservation(customer, date, people)

class Restaurant:
    def __init__(self, name: str, city: str, reservation_system: ReservationSystem) -> None:
        self.name = name
        self.city = city
        self.reservation_system = reservation_system

    def make_reservation(self, customer: str, date: str, people: int) -> None:
        reservation = self.reservation_system.reserve_table(customer, date, people)

        print(reservation)


opentable_api = OpenTableAPI()
opentable_adapter = OpenTableAdapter(opentable_api)

google_api = GoogleReservationsAPI()
google_adapter = GoogleReservationsAdapter(google_api)

restaurant_name = "Pasta Palace"
city = "Santa Marta"
restaurant_opentable = Restaurant(restaurant_name, city, opentable_adapter)
restaurant_google = Restaurant(restaurant_name, city, google_adapter)

restaurant_opentable.make_reservation("Carlos", "2026-09-15", 4)
restaurant_google.make_reservation("Ana", "2026-09-16", 2)


Reserva realizada en OpenTable para Carlos, 4 personas el 2026-09-15
Reserva realizada en Google para Ana, 2 personas el 2026-09-16


## Diagrama UML
```plantuml
@startuml
interface ReservationSystem {
    + reserve_table(customer: str, date: str, people: int)
}
class OpenTableAPI {
    + book_table(customer_name: str, reservation_date: str, guests: int)
}
class GoogleReservationsAPI {
    + make_reservation(name: str, date: str, number_of_people: int)
}
class OpenTableAdapter {
    - api: OpenTableAPI
    + reserve_table(customer: str, date: str, people: int)
}
class GoogleReservationsAdapter {
    - api: GoogleReservationsAPI
    + reserve_table(customer: str, date: str, people: int)
}
class Restaurant {
    - name: str
    - city: str
    - reservation_system: ReservationSystem
    + make_reservation(customer: str, date: str, people: int)
}
ReservationSystem <|.. OpenTableAdapter
ReservationSystem <|.. GoogleReservationsAdapter
OpenTableAdapter --> OpenTableAPI : adapts
GoogleReservationsAdapter --> GoogleReservationsAPI : adapts
Restaurant --> ReservationSystem : uses
@enduml
```

https://www.plantuml.com/plantuml/duml/hPBDJiCm3CVlUOeSXw1xW0gXSKASGC0zSbjlgfQFAdQG9k3TQQTIt8MA7PPJBDk_lp-y8OR8oLdLUiQuXHhr2nB6T-0s-DS3CJhzeNJ_hdKyUj0mL1PNTI8E3cEfYUEDRe1n_7OOEjiRFVDAVQdQ0f5-wj2_3OdtpuyJiGfXVu8p7jmFAFwMOWH_bv2OJlWF8UmiYk992ZdOen6ubL0HP9zSXT64hVcdXmOwnZZY2mrTMyuwlCJ4yex-bCt3BgJV8nbI1C-Ju3IrqQvRFaXRBeoEnrFczxImalFBoo_qzdEzphQp4BSyLvNjjoZkveTRQXXSf7A8pCczisGnsI4whxFKYP3K1dsJdFq6

La elección de aplicar el patrón Adapter se debe a que el problema principal es la incompatibilidad entre interfaces. El restaurante necesita trabajar con una interfaz interna de reservas pero los sistemas externos tienen métodos diferentes. Esto obligaría a modificar las API's pero pueden ser sistemas de terceros que no controlemos. Por este motivo, resulta pertinente crear adaptadores que se encarguen de convertir la información de cada API externa al formato que utiliza el sistema interno de reservas, evitando así tener que modificar el funcionamiento del sistema cada vez que se integre un nuevo proveedor.